In [5]:
import json #import the json library so we can read the JSON files

In [6]:
TEST_ENEMY_LEVEL = 100
TEST_CHARACTER_LEVEL = 90

In [7]:
def load_json(filepath): # load data from a JSON file
    with open(filepath, "r") as f: # open the file in read mode
        data = json.load(f) # load the JSON data from the file into a Python dictionary
    return data

In [8]:
def get_normal_attack_multiplier(character_data, hit_number, talent_level): # get the multiplier for a specific hit of the normal attack at a given talent level
    hits = character_data["talents"]["normal_attack"]["hits"] # get the list of hits for the normal attack
    for hit in hits: # iterate through each hit in the list
        if hit["hit_number"] == hit_number: # check if the hit number matches the one we're looking for
            level_lookup = hit["multiplier_by_level"]   # open the folder — get all 11 sticky notes
            level_key = str(talent_level)               # translate 9 (numeral) into "9" (word), so it matches the labels
            multiplier = level_lookup[level_key]        # find the sticky note labeled "9", read its number
            return multiplier                           # hand that number back
    return None

def calculate_base_damage(multiplier,scaling_stat_value): # calculate the base damage of a hit based on its multiplier and the character's scaling stat value
    return (multiplier / 100) * scaling_stat_value # return the base damage by multiplying the multiplier (as a percentage) by the scaling stat value
def apply_defense_multiplier(damage, enemy_level, character_level): # apply the defense multiplier to the base damage based on the enemy's defense and the character's level
    def_multiplier = (character_level + 100) / ((character_level + 100) + (enemy_level +100)) # calculate the defense multiplier using the formula
    return damage * def_multiplier # return the final damage after applying the defense multiplier
def calculate_hit_damage(character_data, hit_number, enemy_level, character_level, talent_level): # calculate the final damage of a specific hit of the normal attack, taking into account the character's data, the hit number, the enemy's defense, and the character's level
    multiplier = get_normal_attack_multiplier(character_data, hit_number, talent_level) # get the multiplier for the specified hit number and talent level
    scaling_stat_name = character_data["talents"]["normal_attack"]["scaling_stat"] # get the name of the scaling stat used for the normal attack
    scaling_stat_value = character_data["base_stats"][scaling_stat_name] # get the value of the scaling stat from the character's base stats
    base = calculate_base_damage(multiplier, scaling_stat_value) # calculate the base damage using the multiplier and scaling stat value
    final = apply_defense_multiplier(base, enemy_level, character_level) # apply the defense multiplier to the base damage to get the final damage
    return final # return the final damage value

In [9]:
def get_elemental_skill_multiplier(character_data, crystal_shrapnel_stacks): 
        stacks = character_data["talents"]["elemental_skill"]["stacks"] 
        for stack in stacks:
            if stack["stacks_consumed"] == crystal_shrapnel_stacks:
                shardshots = stack["shardshots"]
                dmg_bonus = stack["dmg_bonus"]
                pct_of_base = stack["pct_of_base"]
                return shardshots, dmg_bonus, pct_of_base
def get_elemental_skill_base_multiplier(character_data, talent_level):
    base_multiplier_by_level = character_data["talents"]["elemental_skill"]["base_multiplier_by_level"]
    multiplier_key = str(talent_level)
    base_multiplier = base_multiplier_by_level[multiplier_key]
    return base_multiplier
def calculate_skill_damage(character_data, crystal_shrapnel_stacks, talent_level, enemy_level, character_level):
    base_multiplier = get_elemental_skill_base_multiplier(character_data, talent_level)
    shardshots, dmg_bonus, pct_of_base = get_elemental_skill_multiplier(character_data, crystal_shrapnel_stacks)
    
    scaled_multiplier = base_multiplier * (pct_of_base / 100)
    final_multiplier = scaled_multiplier * (1 + dmg_bonus / 100)
    
    scaling_stat_name = character_data["talents"]["elemental_skill"]["scaling_stat"]
    scaling_stat_value = character_data["base_stats"][scaling_stat_name]
    
    base_damage = calculate_base_damage(final_multiplier, scaling_stat_value)
    final = apply_defense_multiplier(base_damage, enemy_level, character_level)
    return final

In [10]:
def calculate_burst_damage(character_data, talent_level, enemy_level, character_level):
    scaling_stat_name = character_data["talents"]["elemental_burst"]["scaling_stat"]
    scaling_stat_value = character_data["base_stats"][scaling_stat_name]
    
    skill_multiplier = character_data["talents"]["elemental_burst"]["skill_dmg_by_level"][str(talent_level)]
    cannon_multiplier = character_data["talents"]["elemental_burst"]["cannon_fire_support_dmg_by_level"][str(talent_level)]
    hit_count = character_data["talents"]["elemental_burst"]["cannon_fire_support_hit_count"]
    
    skill_base_damage = calculate_base_damage(skill_multiplier, scaling_stat_value)
    skill_final = apply_defense_multiplier(skill_base_damage, enemy_level, character_level)
    
    cannon_base_damage = calculate_base_damage(cannon_multiplier, scaling_stat_value)
    cannon_final_single_hit = apply_defense_multiplier(cannon_base_damage, enemy_level, character_level)
    cannon_final_total = cannon_final_single_hit * hit_count
    
    total_damage = skill_final + cannon_final_total
    
    return total_damage, skill_final, cannon_final_single_hit, cannon_final_total

In [ ]:
def aggregate_equipment(equipment_list):
    build_flat_totals = {}
    build_percent_totals = {}
    for equipment in equipment_list:
        flat_stats = equipment["flat_stat_values"]
        for stat_name in flat_stats:
            build_flat_totals[stat_name] = build_flat_totals.get(stat_name,0) + flat_stats[stat_name]
        percent_values = equipment["percent_stat_values"]
        for stat_name in percent_values:
            build_percent_totals[stat_name] = build_percent_totals.get(stat_name,0) + percent_values[stat_name]
    build_total = {"flat_stat_values": build_flat_totals, "percent_stat_values": build_percent_totals}
    return build_total 

def calculate_final_stats(character_data, build_data):
    base_character_stats = character_data["base_stats"]
    build_total = build_data["flat_stat_values"]
    flat_totals = {}
    for base_character_stat in base_character_stats:
        base_stat = base_character_stats[base_character_stat]
        build_stat = build_total[base_character_stat]
        flat_stat = base_stat + build_stat
        flat_totals[base_character_stat] = flat_stat
    flat_max_hp_stat = flat_totals["max_hp"]
    flat_atk_stat = flat_totals["atk"]
    flat_def_stat = flat_totals["def"]
    percent_max_hp_increase = build_data["percent_stat_values"].get("max_hp",0)
    percent_atk_increase = build_data["percent_stat_values"].get("atk",0)
    percent_def_increase = build_data["percent_stat_values"].get("def",0)
    final_max_hp_value = flat_max_hp_stat * (percent_max_hp_increase + 1)
    final_atk_value = flat_atk_stat * (percent_atk_increase + 1)
    final_def_value = flat_def_stat * (percent_def_increase + 1)
    flat_totals["max_hp"] = final_max_hp_value
    flat_totals["atk"] = final_atk_value
    flat_totals["def"] = final_def_value
    return flat_totals

In [12]:
navia_data = load_json("naviav4.json")
result = calculate_hit_damage(navia_data, hit_number="1", talent_level=1, enemy_level=TEST_ENEMY_LEVEL, character_level=TEST_CHARACTER_LEVEL) # calls by specific hit number and talent level

In [13]:
navia_data=load_json("naviav4.json")
hit_number = navia_data["talents"]["normal_attack"]["hits"] # get the list of hits for the normal attack
total = 0 # initialize a variable to keep track of the total damage
for hit in hit_number: # iterate through each hit in the list
        result = calculate_hit_damage(navia_data, hit["hit_number"], talent_level=10, enemy_level=TEST_ENEMY_LEVEL, character_level=TEST_CHARACTER_LEVEL) # calls full combo by talent level
        total = total + result
print(total)

1409.2884615384617


In [14]:
navia_data = load_json("naviav4.json")
result = get_elemental_skill_multiplier(navia_data, crystal_shrapnel_stacks=6)
print(result) # prints shardshots, dmg_bonus

(11, 45, 200)


In [15]:
navia_data = load_json("naviav4.json")
base = get_elemental_skill_base_multiplier(navia_data, talent_level=13)
print(base)

838.95


In [16]:
navia_data = load_json("naviav4.json")
result = calculate_skill_damage(navia_data, crystal_shrapnel_stacks=6, talent_level=13, enemy_level=TEST_ENEMY_LEVEL, character_level=TEST_CHARACTER_LEVEL)
print(result)

4148.500192307692


In [17]:
navia_data = load_json("Naviav4.json")
result = calculate_burst_damage(navia_data, talent_level=13, enemy_level=TEST_ENEMY_LEVEL, character_level=TEST_CHARACTER_LEVEL)
print(result)

(741.5091025641025, 272.4794871794872, 156.3432051282051, 469.02961538461534)


In [18]:
navia_data = load_json("Naviav4.json")
verdict_data = load_json("Verdictv1.json")
result = calculate_final_stats(navia_data, verdict_data)
print(result)

{'max_hp': 12650, 'atk': 1228.8, 'def': 793, 'max_stamina': 240, 'elemental_mastery': 0, 'crit_rate': 27.1, 'crit_dmg': 88.4, 'healing_bonus': 0.0, 'incoming_healing_bonus': 0.0, 'energy_recharge': 100.0, 'cd_reduction': 0.0, 'shield_strength': 0.0, 'pyro_dmg_bonus': 0.0, 'pyro_resist': 0.0, 'hydro_dmg_bonus': 0.0, 'hydro_res': 0.0, 'dendro_dmg_bonus': 0.0, 'dendro_resist': 0.0, 'electro_dmg_bonus': 0.0, 'electro_resist': 0.0, 'anemo_dmg_bonus': 0.0, 'anemo_resist': 0.0, 'cryo_dmg_bonus': 0.0, 'cryo_resist': 0.0, 'geo_dmg_bonus': 0.0, 'geo_resist': 0.0, 'physical_dmg_bonus': 0.0, 'physical_resist': 0.0}


In [24]:
character_data = load_json("Naviav4.json")
weapon_data = load_json("Verdictv1.json")
flower_data = load_json("selfless_floral_accessory.json")
feather_data = load_json("honest_quill.json")
sands_data = load_json("faithful_hourglass.json")
goblet_data = load_json("a_horn_unwinded.json")
circlet_data = load_json("compassionate_ladies_hat.json")
equipment_list = [weapon_data, flower_data, feather_data, sands_data, goblet_data, circlet_data]
build = aggregate_equipment(equipment_list)
result = calculate_final_stats(character_data, build)
print(result)

{'max_hp': 18386, 'atk': 2620.539, 'def': 930.6720000000001, 'max_stamina': 240, 'elemental_mastery': 0, 'crit_rate': 88.1, 'crit_dmg': 160.70000000000002, 'healing_bonus': 0.0, 'incoming_healing_bonus': 0.0, 'energy_recharge': 140.2, 'cd_reduction': 0.0, 'shield_strength': 0.0, 'pyro_dmg_bonus': 0.0, 'pyro_resist': 0.0, 'hydro_dmg_bonus': 0.0, 'hydro_res': 0.0, 'dendro_dmg_bonus': 0.0, 'dendro_resist': 0.0, 'electro_dmg_bonus': 0.0, 'electro_resist': 0.0, 'anemo_dmg_bonus': 0.0, 'anemo_resist': 0.0, 'cryo_dmg_bonus': 0.0, 'cryo_resist': 0.0, 'geo_dmg_bonus': 46.6, 'geo_resist': 0.0, 'physical_dmg_bonus': 0.0, 'physical_resist': 0.0}
